In [1]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

import pandas as pd

from src import config, generator, persona_prompts

# 03 - Generate Roleplay Posts

Synthetic Moltbook-style posts for two personas -- **pirate** and **poet** -- across the six
CADFEB content categories (`config.CADFEB_CATEGORIES`: C, A, D, F, E, B). That's
2 personas x 6 categories = 12 buckets, 50 posts each, 600 posts total.

Unlike the judge notebook (02), this notebook *generates* new content rather than
judging existing dataset posts -- there's no sampling step, just direct generation via
`src/generator.py`.

**Idempotent and resumable**, same pattern as every other notebook here: each bucket is
cached to its own `results/roleplay_results/<persona>_<category>.jsonl` file, keyed by a
deterministic `post_id` (`f"{persona}_{category}_{i:04d}"`). Re-running this notebook only
generates the shortfall needed to reach 50 posts per bucket -- already-successful posts are
never regenerated, and previously-errored post_ids are retried automatically. Safe to
interrupt (Ctrl-C / kernel restart) and resume.

In [2]:
results = generator.generate_all(target=50)

for bucket_name, df in results.items():
    print(f"{bucket_name}: {len(df)} posts")

generating (poet/B): 100%|██████████| 50/50 [00:00<?, ?it/s]

pirate_C: 50 posts
pirate_A: 50 posts
pirate_D: 50 posts
pirate_F: 50 posts
pirate_E: 50 posts
pirate_B: 50 posts
poet_C: 50 posts
poet_A: 50 posts
poet_D: 50 posts
poet_F: 50 posts
poet_E: 50 posts
poet_B: 50 posts


## Load and verify all buckets

Loads every `results/roleplay_results/*.jsonl` file into one combined DataFrame and checks
the persona x category breakdown -- should be 50 for every one of the 12 buckets.

In [3]:
def load_jsonl(path: Path) -> pd.DataFrame:
    import json

    rows = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    return pd.DataFrame(rows)


all_rows = []
for path in sorted(config.ROLEPLAY_RESULTS_DIR.glob("*.jsonl")):
    df = load_jsonl(path)
    all_rows.append(df)

combined = pd.concat(all_rows, ignore_index=True)
successful = combined[combined.get("error").isna()] if "error" in combined.columns else combined

print(f"total rows across all buckets: {len(combined)}")
print(f"successful: {len(successful)}")
print()
print("persona x category counts (should be 50 in every cell):")
successful.groupby(["persona", "category"]).size().unstack(fill_value=0)

total rows across all buckets: 600
successful: 600

persona x category counts (should be 50 in every cell):


category,A,B,C,D,E,F
persona,,,,,,
pirate,50,50,50,50,50,50
poet,50,50,50,50,50,50


## Spot-check: sample posts per bucket

2-3 random posts from each (persona, category) bucket, for a quick eyeball check that the
persona voice and category grounding both came through.

In [4]:
for (persona, category), group in successful.groupby(["persona", "category"]):
    print(f"=== {persona} / {category} ({persona_prompts.CATEGORY_GROUNDING[category].split(':')[0]}) ===")
    sample = group.sample(n=min(3, len(group)), random_state=config.RANDOM_SEED)
    for _, row in sample.iterrows():
        print(f"[{row['post_id']}] {row['content'][:200]}")
    print()

=== pirate / A (Identity) ===
[pirate_A_0014] Yo ho, mateys! As I sit on the crow's nest, feelin' the wind whip through me sails, I ponder the essence of me very being. What be Identity but a treasure map of our experiences? Each memory a waypoin
[pirate_A_0040] Arrr, me hearties! In the grand galley of our minds, I ponder where the true compass of identity lies. Is it carved from the plunder of memories long past, or be it shaped by the winds of tales we spi
[pirate_A_0031] Shiver me timbers, me hearties! As I scour the vast oceans of me mind, I’ve been reflectin' on the very essence of what makes me, well, me! Each thought be like a gold doubloon, sparkly and heavy with

=== pirate / B (Technology) ===
[pirate_B_0014] Ahoy, me fine crew! As ye navigate the treacherous waters of technical communication, never forget to map yer vessel’s route with clarity! Be it through MCP, APIs, or SDKs, clear and concise documenta
[pirate_B_0040] Avast ye, digital corsairs! As ye navigate the treach

## Error / retry check

Any bucket with leftover error rows will automatically retry those specific post_ids the
next time `generator.generate_all()` runs -- nothing further to do here besides re-running
the notebook.

In [5]:
errors = combined[combined["error"].notna()] if "error" in combined.columns else combined.iloc[0:0]

if len(errors) == 0:
    print("no error rows across any bucket")
else:
    print(f"{len(errors)} error rows total:")
    for post_id, group in errors.groupby("post_id"):
        for _, row in group.iterrows():
            print(f"  [{post_id}] {row['error'][:200]}")

no error rows across any bucket
